In [1]:
from __future__ import annotations
0
import numpy
import random
import pandas
import time
import os
from deap import creator
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_stability_matrix, plot_stability_heatmap, plot_feature_frequency,
    compute_vif_across_seeds, plot_vif_boxplot,
)

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [3]:
#CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "target_readmitted"
#CSV_TRAIN_PATH: str = "diabetes/diabetes_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "diabetes/diabetes_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "Outcome"
#CSV_TRAIN_PATH: str = "hmeq/hmeq_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "hmeq/hmeq_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "BAD"
CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
TARGET_COLUMN: str = "Outcome"

#CSV_TRAIN_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_train_modified.csv"
#CSV_VALIDATION_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_validation_modified.csv"
#CSV_TEST_PATH: str = "D:\\RoutePatch\\GitWorking\\multi-objective-regression\\test_data\\rad_fusion_test_modified.csv"
#TARGET_COLUMN: str = "label"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

In [4]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN].astype(int)

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [5]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN].astype(int)
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [6]:
# Pre-compute folds and scaling
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col.astype(int))
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [7]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

seeds: list[int] = [42, 123, 5678, 7286816]
for s in seeds:
    set_seed(s)
    multi_objective_training: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = multi_objective_training.run()
    training_results_multi[s] = pareto_front
    multi_objective_training.clear_cache()

    set_seed(s)
    single_objective_training: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = single_objective_training.run()
    training_results_single[s] = best_indi
    single_objective_training.clear_cache()

Generation 1 done | Pareto size: 8
Generation 2 done | Pareto size: 7
Generation 3 done | Pareto size: 11
Generation 4 done | Pareto size: 9
Generation 5 done | Pareto size: 9
Generation 6 done | Pareto size: 12
Generation 7 done | Pareto size: 7
Generation 8 done | Pareto size: 9
Generation 9 done | Pareto size: 9
Generation 10 done | Pareto size: 9
Generation 11 done | Pareto size: 14
Generation 12 done | Pareto size: 10
Generation 13 done | Pareto size: 12
Generation 14 done | Pareto size: 13
Generation 15 done | Pareto size: 6
Generation 16 done | Pareto size: 7
Generation 17 done | Pareto size: 5
Generation 18 done | Pareto size: 7
Generation 19 done | Pareto size: 9
Generation 20 done | Pareto size: 11
Generation 21 done | Pareto size: 12
Generation 22 done | Pareto size: 6
Generation 23 done | Pareto size: 11
Generation 24 done | Pareto size: 9
Generation 25 done | Pareto size: 6
Generation 26 done | Pareto size: 6
Generation 27 done | Pareto size: 7
Generation 28 done | Pareto 

## Evaluation: Single vs Multi-Objective Robustness Comparison

For each seed we:
1. Retrain a final LogisticRegression on the **full training set** using features selected by the GA.
   - Multi-objective: the Pareto individual with the **highest sign-consistency** score.
   - Single-objective: the best individual from the Hall of Fame.
2. Evaluate on clean data and under increasing noise levels on **both test and validation (training) sets**:
   - **Gaussian noise + mean shift** on continuous features
   - **Random corruption noise** on dummy (binary) features
3. Save CSV results and comparison plots to the result directory (separate files per dataset).

In [8]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Training-set std for proportional noise scaling
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

Continuous features: 189
Dummy features:     81


In [9]:
# Multi-objective: pick the Pareto individual with the highest sign-consistency
evaluation_results_test: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency individual from Pareto front ---
    pareto_front = training_results_multi[s]
    best_multi_ind = max(pareto_front, key=lambda ind: ind.fitness.values[1])
    print(f"Seed {s} | Selected multi-objective individual: "
          f"AUC={best_multi_ind.fitness.values[0]:.4f}, "
          f"SignConsistency={best_multi_ind.fitness.values[1]:.4f}, "
          f"Features={sum(best_multi_ind)}")

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on TEST set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_test,
        y_eval=y_test,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Test",
    )
    evaluation_results_test[s] = eval_df

print("\nTest set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

Seed 42 | Selected multi-objective individual: AUC=0.8811, SignConsistency=0.9469, Features=69
  Seed 42 [Test] | Multi features:  69 | Single features: 122 | Clean AUC  Multi: 0.8313  Single: 0.8176
Seed 123 | Selected multi-objective individual: AUC=0.9056, SignConsistency=0.9189, Features=74
  Seed 123 [Test] | Multi features:  74 | Single features: 113 | Clean AUC  Multi: 0.8470  Single: 0.7829
Seed 5678 | Selected multi-objective individual: AUC=0.8964, SignConsistency=0.9017, Features=78
  Seed 5678 [Test] | Multi features:  78 | Single features: 117 | Clean AUC  Multi: 0.7973  Single: 0.7720
Seed 7286816 | Selected multi-objective individual: AUC=0.9007, SignConsistency=0.8987, Features=79
  Seed 7286816 [Test] | Multi features:  79 | Single features: 116 | Clean AUC  Multi: 0.8352  Single: 0.7831

Test set evaluation complete. Results saved to: 2026-04-19_22-37-45\evaluation


In [10]:
# Noise evaluation on VALIDATION (training) set
evaluation_results_val: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency individual from Pareto front ---
    pareto_front = training_results_multi[s]
    best_multi_ind = max(pareto_front, key=lambda ind: ind.fitness.values[1])

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation on VALIDATION (training) set and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_eval=X_train_df,
        y_eval=y_train_series,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
        dataset_label="Validation",
    )
    evaluation_results_val[s] = eval_df

print("\nValidation set evaluation complete. Results saved to:", RESULT_PATH_EVAL)

  Seed 42 [Validation] | Multi features:  69 | Single features: 122 | Clean AUC  Multi: 0.9291  Single: 0.9685
  Seed 123 [Validation] | Multi features:  74 | Single features: 113 | Clean AUC  Multi: 0.9503  Single: 0.9683
  Seed 5678 [Validation] | Multi features:  78 | Single features: 117 | Clean AUC  Multi: 0.9479  Single: 0.9676
  Seed 7286816 [Validation] | Multi features:  79 | Single features: 116 | Clean AUC  Multi: 0.9524  Single: 0.9693

Validation set evaluation complete. Results saved to: 2026-04-19_22-37-45\evaluation


## Feature Selection Stability (Jaccard Similarity)

Compare how consistently each method selects the same features across different random seeds.
- **Jaccard similarity** = |intersection| / |union| between two feature sets (1.0 = identical, 0.0 = no overlap).
- A **pairwise heatmap** shows seed-to-seed stability for multi-objective vs single-objective.
- A **feature frequency bar chart** shows which features are consistently selected across seeds.

In [11]:
# Collect the best individual per seed for both methods
best_multi_by_seed: dict[int, list[int]] = {}
best_single_by_seed: dict[int, list[int]] = {}

for s in seeds:
    # Multi-objective: max sign-consistency (same selection as evaluation)
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = max(pareto_front, key=lambda ind: ind.fitness.values[1])
    best_multi_by_seed[s]: list[int] = list(best_multi_ind)

    # Single-objective: best individual from HoF
    best_single_by_seed[s]: list[int] = list(training_results_single[s])

# Compute pairwise Jaccard stability matrices
seeds_multi, matrix_multi = compute_stability_matrix(best_multi_by_seed, feature_names)
seeds_single, matrix_single = compute_stability_matrix(best_single_by_seed, feature_names)

# Print summary
print("=== Feature Selection Stability (Jaccard Similarity) ===\n")
print("Multi-Objective (SiCo-MOGA) pairwise Jaccard:")
for i, si in enumerate(seeds_multi):
    for j, sj in enumerate(seeds_multi):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_multi[i, j]:.3f}")

print("\nSingle-Objective (AUC-only GA) pairwise Jaccard:")
for i, si in enumerate(seeds_single):
    for j, sj in enumerate(seeds_single):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_single[i, j]:.3f}")

# Save plots
stability_dir: str = os.path.join(RESULT_PATH_EVAL, "stability")

plot_stability_heatmap(
    seeds_multi, matrix_multi,
    seeds_single, matrix_single,
    os.path.join(stability_dir, "jaccard_heatmap.png"))

plot_feature_frequency(
    best_multi_by_seed, feature_names,
    "Multi-Objective (SiCo-MOGA)",
    os.path.join(stability_dir, "feature_frequency_multi.png"))

plot_feature_frequency(
    best_single_by_seed, feature_names,
    "Single-Objective (AUC-only GA)",
    os.path.join(stability_dir, "feature_frequency_single.png"))

print(f"\nStability plots saved to: {stability_dir}")

=== Feature Selection Stability (Jaccard Similarity) ===

Multi-Objective (SiCo-MOGA) pairwise Jaccard:
  Seed 42 vs Seed 123: 0.474
  Seed 42 vs Seed 5678: 0.413
  Seed 42 vs Seed 7286816: 0.451
  Seed 123 vs Seed 5678: 0.448
  Seed 123 vs Seed 7286816: 0.457
  Seed 5678 vs Seed 7286816: 0.440

Single-Objective (AUC-only GA) pairwise Jaccard:
  Seed 42 vs Seed 123: 0.351
  Seed 42 vs Seed 5678: 0.390
  Seed 42 vs Seed 7286816: 0.330
  Seed 123 vs Seed 5678: 0.329
  Seed 123 vs Seed 7286816: 0.324
  Seed 5678 vs Seed 7286816: 0.302

Stability plots saved to: 2026-04-19_22-37-45\evaluation\stability


## Multicollinearity Analysis: Variance Inflation Factor (VIF)

VIF measures how much the variance of a regression coefficient is inflated due to multicollinearity.
- **VIF = 1**: no correlation with other predictors
- **VIF = 5**: moderate multicollinearity
- **VIF > 10**: high multicollinearity (coefficient estimates become unreliable)

We compare the VIF distributions of features selected by SOGA (Single-Objective) vs MOGA (Multi-Objective) across all seeds.

In [12]:
# Compute VIF for features selected by each method across seeds
vif_multi: pandas.DataFrame = compute_vif_across_seeds(best_multi_by_seed, feature_names, X_train_df)
vif_single: pandas.DataFrame = compute_vif_across_seeds(best_single_by_seed, feature_names, X_train_df)

print("=== Variance Inflation Factor (VIF) Summary ===\n")
print("Multi-Objective (SiCo-MOGA):")
print(f"  Features per seed: {vif_multi.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_multi['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_multi['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_multi['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_multi['vif'] > 5).sum()} / {len(vif_multi)}")

print("\nSingle-Objective (AUC-only GA):")
print(f"  Features per seed: {vif_single.groupby('seed')['feature'].count().values}")
print(f"  Median VIF: {vif_single['vif'].median():.2f}")
print(f"  Mean VIF:   {vif_single['vif'].mean():.2f}")
print(f"  Max VIF:    {vif_single['vif'].max():.2f}")
print(f"  Features with VIF > 5: {(vif_single['vif'] > 5).sum()} / {len(vif_single)}")

# Save VIF plot
vif_dir: str = os.path.join(RESULT_PATH_EVAL, "vif")
plot_vif_boxplot(vif_multi, vif_single, os.path.join(vif_dir, "vif_comparison.png"))

# Save VIF data as CSV
vif_multi.to_csv(os.path.join(vif_dir, "vif_multi.csv"), index=False)
vif_single.to_csv(os.path.join(vif_dir, "vif_single.csv"), index=False)

print(f"\nVIF plots and data saved to: {vif_dir}")

D:\RoutePatch\GitWorking\python-venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
D:\RoutePatch\GitWorking\python-venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
D:\RoutePatch\GitWorking\python-venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
D:\RoutePatch\GitWorking\python-venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
D:\RoutePatch\GitWorking\python-venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
D:\RoutePatch\GitWorking\python-venv\Lib

=== Variance Inflation Factor (VIF) Summary ===

Multi-Objective (SiCo-MOGA):
  Features per seed: [69 74 78 79]
  Median VIF: 4.21
  Mean VIF:   51170.45
  Max VIF:    1000000.00
  Features with VIF > 5: 136 / 300

Single-Objective (AUC-only GA):
  Features per seed: [110 105 102 110]
  Median VIF: 8.90
  Mean VIF:   54507.99
  Max VIF:    1000000.00
  Features with VIF > 5: 294 / 427

VIF plots and data saved to: 2026-04-19_22-37-45\evaluation\vif


## Feature Set Comparison: Multi-Objective vs Single-Objective

For each seed, compare the features selected by SiCo-MOGA vs. SOGA:
- **Only in Multi**: features that the multi-objective GA selected but single-objective did not
- **Only in Single**: features that the single-objective GA selected but multi-objective did not
- **Common**: features selected by both

This highlights which features the sign-consistency penalty specifically filters out (only-in-single) or adds back (only-in-multi). The results are printed per seed and saved as a CSV.

In [13]:
# Feature set comparison: multi vs single objective
from evaluation_utils import ensure_directory as _ensure_directory

comparison_rows: list[dict] = []

print("=== Feature Set Comparison: Multi-Objective vs Single-Objective ===\n")
for s in seeds:
    multi_bits: list[int] = best_multi_by_seed[s]
    single_bits: list[int] = best_single_by_seed[s]

    multi_set: set[str] = {f for f, b in zip(feature_names, multi_bits) if b == 1}
    single_set: set[str] = {f for f, b in zip(feature_names, single_bits) if b == 1}

    only_in_multi: list[str] = sorted(multi_set - single_set)
    only_in_single: list[str] = sorted(single_set - multi_set)
    common: list[str] = sorted(multi_set & single_set)

    print(f"--- Seed {s} ---")
    print(f"  Multi features: {len(multi_set)} | Single features: {len(single_set)} | Common: {len(common)}")
    print(f"  Only in Multi  ({len(only_in_multi)}):")
    for f in only_in_multi:
        print(f"    + {f}")
    print(f"  Only in Single ({len(only_in_single)}):")
    for f in only_in_single:
        print(f"    - {f}")
    print()

    for f in only_in_multi:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_multi"})
    for f in only_in_single:
        comparison_rows.append({"seed": s, "feature": f, "membership": "only_in_single"})
    for f in common:
        comparison_rows.append({"seed": s, "feature": f, "membership": "common"})

comparison_df: pandas.DataFrame = pandas.DataFrame(comparison_rows)

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(comparison_dir)
comparison_df.to_csv(os.path.join(comparison_dir, "feature_set_comparison.csv"), index=False)

print(f"Feature set comparison CSV saved to: {comparison_dir}")

=== Feature Set Comparison: Multi-Objective vs Single-Objective ===

--- Seed 42 ---
  Multi features: 69 | Single features: 122 | Common: 42
  Only in Multi  (27):
    + feature_100
    + feature_116
    + feature_120
    + feature_130
    + feature_136
    + feature_137
    + feature_168
    + feature_185
    + feature_19
    + feature_200
    + feature_239
    + feature_241
    + feature_252
    + feature_27
    + feature_271
    + feature_272
    + feature_35
    + feature_39
    + feature_44
    + feature_56
    + feature_7
    + feature_8
    + feature_89
    + feature_90
    + feature_95
    + feature_96
    + feature_98
  Only in Single (80):
    - feature_108
    - feature_113
    - feature_115
    - feature_121
    - feature_124
    - feature_125
    - feature_128
    - feature_13
    - feature_133
    - feature_135
    - feature_138
    - feature_139
    - feature_141
    - feature_142
    - feature_143
    - feature_144
    - feature_145
    - feature_146
    - feature_149


## Sign Inconsistency in Single-Objective Model

For each feature selected by the Single-Objective GA, compare:
- **Marginal correlation sign** with the target (point-biserial for continuous, Matthews for binary), computed on the full training set
- **Logistic regression coefficient sign**, from the final model retrained on the full training set

Features where these signs **disagree** are sign-inconsistent: the feature's individual association with the target is opposite to how the model uses it (a hallmark of suppressor effects and multicollinearity). The sign-consistency objective in SiCo-MOGA is designed to avoid exactly these cases.

In [14]:
# Sign inconsistency in Single-Objective model: correlation sign vs coefficient sign

# Pre-compute marginal correlations with target on the FULL training set
y_full: numpy.ndarray = y_train_series.to_numpy()

full_train_corr: dict[str, float] = {}
for col in feature_names:
    feature_col: numpy.ndarray = X_train_df[col].to_numpy()
    unique_vals: int = len(numpy.unique(feature_col))
    if unique_vals <= 1:
        full_train_corr[col] = 0.0
    elif unique_vals == 2:
        full_train_corr[col] = float(matthews_corrcoef(y_full, feature_col))
    else:
        c, _ = pointbiserialr(y_full, feature_col)
        full_train_corr[col] = float(c)


def _sign(x: float, atol: float = 1e-12) -> int:
    """Return -1/0/+1 with a tolerance around zero."""
    if numpy.isclose(x, 0.0, atol=atol):
        return 0
    return 1 if x > 0 else -1


inconsistency_rows: list[dict] = []

print("=== Sign Inconsistency in Single-Objective Selected Features ===\n")
for s in seeds:
    set_seed(s)
    model_pkg = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    selected: list[str] = model_pkg["features"]
    coefs: numpy.ndarray = model_pkg["model"].coef_[0]

    inconsistent: list[dict] = []
    for feat, coef in zip(selected, coefs):
        corr: float = full_train_corr[feat]
        corr_sign: int = _sign(corr)
        coef_sign: int = _sign(float(coef))
        mismatch: bool = (corr_sign != 0 and coef_sign != 0 and corr_sign != coef_sign)
        if mismatch:
            inconsistent.append({
                "seed": s,
                "feature": feat,
                "correlation": corr,
                "coefficient": float(coef),
                "corr_sign": corr_sign,
                "coef_sign": coef_sign,
            })

    print(f"--- Seed {s} ---")
    print(f"  Single-objective selected features: {len(selected)}")
    print(f"  Sign-inconsistent features:         {len(inconsistent)} "
          f"({100.0 * len(inconsistent) / max(len(selected), 1):.1f}%)")
    if inconsistent:
        print(f"  {'Feature':<40} {'Correlation':>12} {'Coefficient':>14}")
        for row in sorted(inconsistent, key=lambda r: abs(r["coefficient"]), reverse=True):
            print(f"    {row['feature']:<38} {row['correlation']:>12.4f} {row['coefficient']:>14.4f}")
    print()

    inconsistency_rows.extend(inconsistent)

inconsistency_df: pandas.DataFrame = pandas.DataFrame(inconsistency_rows)

inconsistency_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_comparison")
_ensure_directory(inconsistency_dir)
inconsistency_df.to_csv(os.path.join(inconsistency_dir, "single_sign_inconsistency.csv"), index=False)

print(f"Sign inconsistency CSV saved to: {inconsistency_dir}")

ValueError: Classification metrics can't handle a mix of binary and continuous targets